# 📌 PinPoint AI — PDF Understanding & RAG Pipeline

PinPoint AI converts a PDF/book into searchable knowledge and prepares the content for summaries, question answering, and visual storytelling.

This notebook implements the core document-processing pipeline:
PDF extraction → cleaning → chunking → embeddings → semantic retrieval.


In [ ]:
from pathlib import Path
import re
import numpy as np

import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import faiss

PDF_PATH = Path("data/sample.pdf")


## 1. Extract Text


In [ ]:
if not PDF_PATH.exists():
    raise FileNotFoundError("Add a PDF as data/sample.pdf")

doc = fitz.open(PDF_PATH)

pages = []
for page_num, page in enumerate(doc, start=1):
    text = page.get_text("text")
    pages.append({
        "page": page_num,
        "text": text
    })

print("Pages extracted:", len(pages))
print(pages[0]["text"][:1000])


## 2. Clean Text


In [ ]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

clean_pages = [
    {"page": p["page"], "text": clean_text(p["text"])}
    for p in pages
]

clean_pages[:2]


## 3. Create Context-Preserving Chunks


In [ ]:
def chunk_text(text, chunk_size=900, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

chunks = []

for page in clean_pages:
    for chunk_id, chunk in enumerate(chunk_text(page["text"])):
        if chunk.strip():
            chunks.append({
                "page": page["page"],
                "chunk_id": chunk_id,
                "text": chunk
            })

print("Chunks:", len(chunks))


## 4. Generate Embeddings


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings, dtype="float32")
print("Embedding shape:", embeddings.shape)


## 5. Build FAISS Vector Index


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Vectors stored:", index.ntotal)


## 6. Semantic Search


In [ ]:
def search(query, top_k=5):
    query_vector = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(query_vector, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        item = chunks[int(idx)].copy()
        item["score"] = float(score)
        results.append(item)

    return results

results = search("What is the main idea of this document?")

for r in results:
    print(f"Page {r['page']} | score={r['score']:.3f}")
    print(r["text"][:500])
    print("-" * 80)


## 7. RAG / Generation Layer

The retrieved chunks can now be passed to a generative language model for:

- Context-grounded question answering
- Summaries
- Story generation
- Scene descriptions
- Visual prompts
- Narration scripts

Keep the generated answer grounded in retrieved document context and clearly separate generated content from source text.


## 8. Future Modules

- Chapter detection
- Chroma vector database
- OCR for scanned PDFs
- Image generation
- Text-to-speech narration
- Multilingual support
- Multi-document search
- Streamlit deployment
